# Semana 8 — Montando um Mini-GPT
**Fase 2: Deep Learning e o Ecossistema Hugging Face** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/applied-ai-architecture/blob/main/02-Phase-2-Deep-Learning/notebooks/semana_08_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-08-Conceitos.md](https://github.com/NetoRibeiro/applied-ai-architecture/blob/main/02-Phase-2-Deep-Learning/docs/Semana-08-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_08_gabarito.ipynb`.

> **GPU:** em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecione **T4 GPU** antes de começar.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math, time, matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(1337); print(device)

## Passo 1 — Dados (mesmo corpus e tokenizer da Semana 6)

In [ ]:
# --- Corpus: tenta baixar o Tiny Shakespeare (1 MB); sem internet, usa um texto curto embutido ---
import urllib.request
URL_CORPUS = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
try:
    texto = urllib.request.urlopen(URL_CORPUS, timeout=15).read().decode("utf-8")
except Exception as e:
    print("Download indisponível, usando texto embutido:", type(e).__name__)
    texto = ("""Um modelo de linguagem aprende a prever o próximo símbolo a partir dos símbolos anteriores.
Quando o texto é pequeno, o modelo decora; quando o texto é grande, o modelo generaliza.
A arquiteta de dados perguntou qual era a métrica correta, e o engenheiro respondeu que dependia do custo de cada erro.
Quem quer entender um sistema precisa construir uma versão pequena dele, quebrar essa versão e consertar de novo.
O vetor que representa uma frase fica perto do vetor de outra frase quando as duas querem dizer quase a mesma coisa.
A cada passo o gradiente aponta a direção em que o erro cresce, e o otimizador caminha para o lado contrário.
Que o modelo erre é esperado; que ninguém meça o erro é que é o problema de arquitetura.
""" * 40)
# Para treinar em português, troque `texto` pelo conteúdo de qualquer arquivo .txt (ex.: um livro em domínio público).
chars = sorted(set(texto)); V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]; decode = lambda ids: "".join(itos[i] for i in ids)
dados = torch.tensor(encode(texto), dtype=torch.long); n = int(0.9 * len(dados)); treino, val = dados[:n], dados[n:]

# Configuração: maior na GPU, mínima na CPU
cfg = dict(n_embd=192, n_head=6, n_layer=4, block=128, batch=64, passos=3000) if device == "cuda" else \
      dict(n_embd=64, n_head=4, n_layer=2, block=48, batch=32, passos=400)
cfg["block"] = min(cfg["block"], len(val) - 2)

def get_batch(split):
    d = treino if split == "treino" else val
    ix = torch.randint(len(d) - cfg["block"] - 1, (cfg["batch"],))
    x = torch.stack([d[i:i + cfg["block"]] for i in ix]); y = torch.stack([d[i + 1:i + cfg["block"] + 1] for i in ix])
    return x.to(device), y.to(device)
print(cfg, "| vocabulário:", V)

## Passo 2 — Multi-head attention eficiente
Em vez de uma lista de heads, uma única projeção gera `q, k, v` para todas as heads; um `view`/`transpose` separa as heads em uma dimensão extra. `F.scaled_dot_product_attention(..., is_causal=True)` aplica a causal mask e usa a implementação otimizada.

In [ ]:
class Atencao(nn.Module):
    def __init__(self, C, n_head):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(C, 3 * C, bias=False); self.proj = nn.Linear(C, C)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # [SEU CÓDIGO AQUI] separe as heads (view + transpose), aplique a causal attention e junte de volta em (B, T, C)
        return self.proj(y)

## Passo 3 — MLP e Bloco
MLP: `Linear(C, 4C) → GELU → Linear(4C, C)`. Bloco pre-norm com duas residual connections.

In [ ]:
class MLP(nn.Module):
    def __init__(self, C):
        super().__init__()
        # [SEU CÓDIGO AQUI]
    def forward(self, x): return self.rede(x)

class Bloco(nn.Module):
    def __init__(self, C, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(C); self.att = Atencao(C, n_head)
        self.ln2 = nn.LayerNorm(C); self.mlp = MLP(C)
    def forward(self, x):
        # [SEU CÓDIGO AQUI] as duas sublayers com residual (pre-norm)
        return x

## Passo 4 — O GPT

In [ ]:
class GPT(nn.Module):
    def __init__(self, V, n_embd, n_head, n_layer, block, **_):
        super().__init__()
        self.block = block
        self.tok = nn.Embedding(V, n_embd); self.pos = nn.Embedding(block, n_embd)
        self.blocos = nn.ModuleList([Bloco(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.head = nn.Linear(n_embd, V, bias=False)
        self.head.weight = self.tok.weight                       # weight tying
        nn.init.normal_(self.tok.weight, std=0.02); nn.init.normal_(self.pos.weight, std=0.02)
    def forward(self, idx, alvos=None):
        B, T = idx.shape
        # [SEU CÓDIGO AQUI] embeddings de token + posição, blocos, ln_f e head
        loss = None if alvos is None else F.cross_entropy(logits.view(-1, logits.size(-1)), alvos.view(-1))
        return logits, loss
    @torch.no_grad()
    def gerar(self, idx, n_tokens, temperatura=1.0, top_k=None):
        for _ in range(n_tokens):
            logits, _ = self(idx[:, -self.block:])                # recorta ao contexto máximo
            logits = logits[:, -1, :] / temperatura
            if top_k:
                # [SEU CÓDIGO AQUI] top-k: mantenha só os k maiores logits (os demais viram -inf)
            idx = torch.cat([idx, torch.multinomial(F.softmax(logits, -1), 1)], dim=1)
        return idx

modelo = GPT(V, **cfg).to(device)
n_params = sum(p.numel() for p in modelo.parameters())
print(f"parâmetros: {n_params:,}")

## Passo 5 — Confira a conta de parâmetros
Compare o número real com a estimativa `V·d + 12·N·d²` da leitura.

In [ ]:
d, N = cfg["n_embd"], cfg["n_layer"]
# [SEU CÓDIGO AQUI] calcule a estimativa
print(f"estimativa: {estimativa:,} | real: {n_params:,} | diferença: {abs(estimativa - n_params) / n_params:.1%}")

## Passo 6 — Treino

In [ ]:
@torch.no_grad()
def perda_media(split, n=20):
    modelo.eval(); l = sum(modelo(*get_batch(split))[1].item() for _ in range(n)) / n; modelo.train(); return l

opt = torch.optim.AdamW(modelo.parameters(), lr=3e-3 if device == "cpu" else 1e-3, weight_decay=0.01)
hist, t0 = [], time.time()
for p in range(cfg["passos"]):
    # [SEU CÓDIGO AQUI] passo de treino com clip de gradiente (norma máxima 1.0)
    if p % max(1, cfg["passos"] // 8) == 0 or p == cfg["passos"] - 1:
        hist.append((p, perda_media("treino"), perda_media("val"))); print(f"passo {p:5d} | treino {hist[-1][1]:.3f} | val {hist[-1][2]:.3f} | {time.time()-t0:.0f}s")
plt.plot(*zip(*[(h[0], h[1]) for h in hist]), label="treino"); plt.plot(*zip(*[(h[0], h[2]) for h in hist]), label="val"); plt.legend(); plt.show()

## Passo 7 — Generation

In [ ]:
inicio = torch.tensor([encode("\n")], device=device) if "\n" in stoi else torch.zeros((1, 1), dtype=torch.long, device=device)
modelo.eval()
print(decode(modelo.gerar(inicio, 400, temperatura=0.8, top_k=20)[0].tolist()))

Compare com o texto do bigrama da Semana 6. O que melhorou? O que ainda falta?

## Passo 8 — Custo quadrático da attention
Meça o tempo de um forward para contextos crescentes.

In [ ]:
m2 = GPT(V, n_embd=64, n_head=4, n_layer=2, block=2048).to(device).eval()
for T in (128, 256, 512, 1024, 2048):
    x = torch.randint(V, (1, T), device=device)
    with torch.no_grad():
        m2(x); t0 = time.perf_counter(); m2(x)
        if device == "cuda": torch.cuda.synchronize()
    print(f"T={T:5d} | {1000 * (time.perf_counter() - t0):7.1f} ms")

## Passo 9 — Leia um `config.json` de verdade
Use sua fórmula para estimar os parâmetros de um modelo com `hidden_size=896`, `num_hidden_layers=24`, `vocab_size=151936` (um modelo aberto de ~0,5B). Onde está a maior parte dos parâmetros: nos embeddings ou nos blocos? Note que alguns modelos usam MLP com fator diferente de 4.

In [ ]:
# [SEU CÓDIGO AQUI] aplique a fórmula e imprima embeddings, blocos e total